In [8]:
# === query_object.py isolation test (FIXED import for dataclasses) ===

from pathlib import Path
import json
import hashlib
import importlib.util
import sys
from pprint import pprint

Y_PATH = Path("/home/hello/Projects/Statements/runner/output/Y_inferred.json")
QO_PATH = Path("/home/hello/Projects/Statements/code/moltie/schemas/query_object.py")

def load_json(p: Path):
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")
    return json.loads(p.read_text(encoding="utf-8"))

def import_module_from_path(module_path: Path, name: str = "query_object_under_test"):
    if not module_path.exists():
        raise FileNotFoundError(f"Missing module: {module_path}")

    spec = importlib.util.spec_from_file_location(name, str(module_path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not create import spec for: {module_path}")

    mod = importlib.util.module_from_spec(spec)

    # ✅ CRITICAL: register in sys.modules BEFORE exec_module (dataclasses needs this)
    sys.modules[name] = mod

    spec.loader.exec_module(mod)  # type: ignore[attr-defined]
    return mod

def sha256(obj) -> str:
    s = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

# --- Load ---
y_json = load_json(Y_PATH)
qo = import_module_from_path(QO_PATH)

print("✅ Loaded Y:", Y_PATH)
print("✅ Loaded module:", QO_PATH)
print("x_tests:", len((y_json.get("x_tests") or {})))

# ----------------------------
# Test vectors
# ----------------------------
evidence_to_x = {
    "evid_001": ["X1", "X3"],
    "evid_002": ["X2"],
    "evid_003": ["X2", "X4", "X1"],
    "evid_004": [],
    "evid_005": ["  X5  ", "X5"],
}

# ----------------------------
# Run core functions
# ----------------------------
collapsed = qo.collapse_x_hits_one_per_evidence(y_json, evidence_to_x)
xs_unique = qo.unique_x_tests_from_collapsed(collapsed)
merged = qo.merge_indicators_and_excludes(y_json, xs_unique)
atom = qo.build_atom_query_from_hits(
    atom_id="ATOM_TEST_001",
    y_json=y_json,
    evidence_to_x=evidence_to_x,
    proposition=None,
)
deduped_y = qo.build_deduped_y_view(y_json, collapsed)

print("\n=== collapsed (evidence_id -> Xi) ===")
pprint(collapsed)

print("\n=== unique Xi (sorted) ===")
pprint(xs_unique)

print("\n=== merged sizes ===")
print("positive_indicators:", len(merged["positive_indicators"]))
print("excludes:", len(merged["excludes"]))

print("\n=== AtomQuery ===")
atom_d = atom.to_dict() if hasattr(atom, "to_dict") else atom
pprint(atom_d)

# Validator check
qo.AtomQueryValidator.validate(atom_d)

print("\n=== dedup audit ===")
pprint(deduped_y.get("dedup_audit"))

# ----------------------------
# Determinism hash test
# ----------------------------
hashes = []
for _ in range(10):
    c = qo.collapse_x_hits_one_per_evidence(y_json, evidence_to_x)
    a = qo.build_atom_query_from_hits(atom_id="ATOM_TEST_001", y_json=y_json, evidence_to_x=evidence_to_x, proposition=None)
    d = qo.build_deduped_y_view(y_json, c)
    hashes.append(sha256({
        "collapsed": c,
        "atom": a.to_dict(),
        "deduped_keys": sorted((d.get("x_tests") or {}).keys()),
        "audit": d.get("dedup_audit"),
    }))

print("\n=== determinism hashes (must all match) ===")
pprint(hashes)
assert len(set(hashes)) == 1, "❌ Non-deterministic output detected!"

# ----------------------------
# Strict synthetic tie-break test
# ----------------------------
y_json_tie = {"x_tests": {
    "XAAA": {"required_elements": ["a"], "positive_indicators": ["p1"], "excludes": ["e1"]},
    "XBBB": {"required_elements": ["a"], "positive_indicators": ["p1"], "excludes": ["e1"]},
}}
chosen = qo.select_most_specific_x(y_json_tie, ["XBBB", "XAAA"])
print("\n=== tie-break chosen ===", chosen)
assert chosen == "XAAA", "❌ Tie-break failed: expected lexical-min (XAAA)"

print("\n✅ ALL query_object.py tests PASSED")


✅ Loaded Y: /home/hello/Projects/Statements/runner/output/Y_inferred.json
✅ Loaded module: /home/hello/Projects/Statements/code/moltie/schemas/query_object.py
x_tests: 7

=== collapsed (evidence_id -> Xi) ===
{'evid_001': 'X1', 'evid_002': 'X2', 'evid_003': 'X1', 'evid_005': 'X5'}

=== unique Xi (sorted) ===
['X1', 'X2', 'X5']

=== merged sizes ===
positive_indicators: 9
excludes: 6

=== AtomQuery ===
{'atom_id': 'ATOM_TEST_001',
 'excludes': ['recent negative performance reviews',
              'formal warnings',
              'no written communication',
              'full implementation of role changes',
              'consistent application of standards',
              'no different treatment'],
 'expansion_terms': [],
 'keyword_seeds': [],
 'positive_indicators': ['consistently received positive performance '
                         'evaluations',
                         'no formal warnings',
                         'top performer',
                         'written communicati

In [9]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "mistral-small3.2:latest"  # <-- change if needed

payload = {
    "model": MODEL,
    "prompt": "Return a JSON object with keys x (int) and y (string).",
    "stream": False,
    "format": "json",
}

r = requests.post(OLLAMA_URL, json=payload, timeout=60)
print("Status:", r.status_code)
print("Raw response JSON:\n")
print(json.dumps(r.json(), indent=2))


Status: 200
Raw response JSON:

{
  "model": "mistral-small3.2:latest",
  "created_at": "2026-02-15T17:24:12.213303107Z",
  "response": "{\"x\": 0, \"y\": \"example\"}",
  "done": true,
  "done_reason": "stop",
  "context": [
    17,
    4568,
    1584,
    42301,
    2784,
    29121,
    1032,
    1051,
    1046,
    1050,
    1044,
    1261,
    43520,
    26242,
    11512,
    1319,
    23947,
    1077,
    1041,
    6254,
    1536,
    42301,
    2784,
    26554,
    1044,
    1261,
    8689,
    53862,
    3518,
    125609,
    1294,
    6993,
    1626,
    4568,
    4053,
    1420,
    26554,
    27089,
    4418,
    2301,
    38680,
    1626,
    16994,
    7807,
    4469,
    1486,
    3804,
    12220,
    1408,
    1032,
    1050,
    1048,
    1050,
    1051,
    1045,
    1049,
    1048,
    1045,
    1048,
    1049,
    1338,
    7651,
    1636,
    6185,
    1605,
    5257,
    2314,
    2269,
    3686,
    1505,
    2200,
    1278,
    3330,
    1681,
    4546,
    10867,

In [10]:
schema = {
    "type": "object",
    "properties": {
        "a": {"type": "integer"},
        "b": {"type": "string"}
    },
    "required": ["a", "b"],
    "additionalProperties": False
}

payload = {
    "model": MODEL,
    "prompt": "Return a JSON object with a=123 and b='test'.",
    "stream": False,
    "format": schema
}

r = requests.post(OLLAMA_URL, json=payload, timeout=60)

print("Status:", r.status_code)
print("Raw response JSON:\n")
print(json.dumps(r.json(), indent=2))


Status: 200
Raw response JSON:

{
  "model": "mistral-small3.2:latest",
  "created_at": "2026-02-15T17:24:41.913832059Z",
  "response": "{\n  \"a\": 123,\n  \"b\": \"test\"\n}",
  "done": true,
  "done_reason": "stop",
  "context": [
    17,
    4568,
    1584,
    42301,
    2784,
    29121,
    1032,
    1051,
    1046,
    1050,
    1044,
    1261,
    43520,
    26242,
    11512,
    1319,
    23947,
    1077,
    1041,
    6254,
    1536,
    42301,
    2784,
    26554,
    1044,
    1261,
    8689,
    53862,
    3518,
    125609,
    1294,
    6993,
    1626,
    4568,
    4053,
    1420,
    26554,
    27089,
    4418,
    2301,
    38680,
    1626,
    16994,
    7807,
    4469,
    1486,
    3804,
    12220,
    1408,
    1032,
    1050,
    1048,
    1050,
    1051,
    1045,
    1049,
    1048,
    1045,
    1048,
    1049,
    1338,
    7651,
    1636,
    6185,
    1605,
    5257,
    2314,
    2269,
    3686,
    1505,
    2200,
    1278,
    3330,
    1681,
    4546,
  